# 03 — Build Cross-Section

I combine the August 2025 occupancy outcome with meter characteristics, Census neighborhood measures, nearby business activity, and available zoning information. The goal is one row per parking space with candidate predictors for later EDA and variable selection, not a selected regression specification.

## 1. Load the prepared parking data

I use the occupancy outcome and meter variables already saved by Notebooks 01 and 02. I keep their definitions unchanged, including the time-weighted outcome, 24-hour interval rule, 25-active-day requirement, and the price proxy.

In [1]:
import json
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
import geopandas as gpd
from shapely import build_area
from shapely.geometry import MultiLineString

occupancy = pd.read_csv("../../data/processed/space_occupancy_aug2025.csv", dtype={"SpaceID": str})
meters = pd.read_csv("../../data/processed/meter_inventory_clean.csv", dtype={"SpaceID": str})

print("Occupancy rows:", len(occupancy))
print("Inventory rows:", len(meters))
print("Occupancy duplicate IDs:", occupancy["SpaceID"].duplicated().sum())
print("Inventory duplicate IDs:", meters["SpaceID"].duplicated().sum())
occupancy.head()

Occupancy rows: 4185
Inventory rows: 35015
Occupancy duplicate IDs: 0
Inventory duplicate IDs: 0


,SpaceID,occupied_minutes,total_minutes,occupancy_rate,active_days
0,61111,19511.316667,37847.833333,0.515520,30
1,61112,17954.033333,38293.550000,0.468853,30
2,61113,21939.050000,40604.650000,0.540309,30
3,61114,26049.100000,43845.700000,0.594108,31
4,61116,22993.916667,43823.916667,0.524689,31


## 2. Match occupancy to meter characteristics

I first use a left merge to show which occupancy spaces have meter information. The merge check is separate from the sample decision: I inspect the unmatched spaces before establishing the matched base sample.

In [2]:
parking_match = occupancy.merge(meters, on="SpaceID", how="left", indicator=True, validate="one_to_one")
print(parking_match["_merge"].value_counts())
print("Match percentage:", (parking_match["_merge"] == "both").mean() * 100)

unmatched_meters = parking_match.loc[parking_match["_merge"] == "left_only"]
print("Unmatched spaces:", len(unmatched_meters))
unmatched_meters[["SpaceID", "occupancy_rate", "active_days"]].head(20)

_merge
both          4139
left_only       46
right_only       0
Name: count, dtype: int64
Match percentage: 98.90083632019116
Unmatched spaces: 46


,SpaceID,occupancy_rate,active_days
5,61117,0.501990,31
6,61118,0.521684,31
37,61351,0.650139,31
38,61352,0.682325,31
39,61353,0.677709,31
40,61354,0.698670,31
41,61355,0.652248,31
42,61356,0.664612,31
43,61357,0.645580,31
44,61358,0.700899,29


I use the 4,139 matched spaces as the base cross-section because they have the meter characteristics and coordinates needed for the remaining steps. This explicitly leaves out the 46 occupancy spaces without an inventory match; later missing Census or zoning predictors will not automatically remove additional spaces.

In [3]:
parking = parking_match.loc[parking_match["_merge"] == "both"].drop(columns="_merge").copy()
print("Base shape:", parking.shape)
print("Unique SpaceIDs:", parking["SpaceID"].nunique())
print("Duplicate SpaceIDs:", parking["SpaceID"].duplicated().sum())
parking[["SpaceID", "occupancy_rate", "price_midpoint", "latitude", "longitude"]].head()

Base shape: (4139, 14)
Unique SpaceIDs: 4139
Duplicate SpaceIDs: 0


,SpaceID,occupancy_rate,price_midpoint,latitude,longitude
0,61111,0.515520,1.0,33.997217,-118.478897
1,61112,0.468853,1.0,33.997177,-118.478857
2,61113,0.540309,1.0,33.997106,-118.478788
3,61114,0.594108,1.0,33.997058,-118.478741
4,61116,0.524689,1.0,33.996964,-118.478645


## 3. Read the Census estimates by tract

The saved ACS JSON is organized by Census geography, with nested estimates for each table. I extract the agreed variables using the source column codes and retain the 2024 five-year release, which summarizes 2020–2024 rather than August 2025 alone.

In [4]:
acs_path = Path("../../data/raw/census/20261017T052111914317Z/acs2024_5yr_la_county_tracts.json")
with acs_path.open() as f:
    acs_raw = json.load(f)

print("ACS release:", acs_raw["release"])
acs_rows = []
for geography, tables in acs_raw["data"].items():
    acs_rows.append({
        "GEOID": geography.replace("14000US", ""),
        "population": tables["B01003"]["estimate"]["B01003001"],
        "median_income": tables["B19013"]["estimate"]["B19013001"],
        "housing_units": tables["B25001"]["estimate"]["B25001001"],
        "commuters": tables["B08301"]["estimate"]["B08301001"],
        "drive_alone": tables["B08301"]["estimate"]["B08301003"],
        "public_transit": tables["B08301"]["estimate"]["B08301010"],
        "walked": tables["B08301"]["estimate"]["B08301019"]
    })
acs = pd.DataFrame(acs_rows)
print("ACS shape:", acs.shape)
print("Duplicate tract IDs:", acs["GEOID"].duplicated().sum())
print("Missing estimates:")
print(acs.isna().sum())
acs.head()

ACS release: {'id': 'acs2024_5yr', 'name': 'ACS 2024 5-year', 'years': '2020-2024'}
ACS shape: (2498, 8)
Duplicate tract IDs: 0
Missing estimates:
GEOID              0
population         0
median_income     42
housing_units      0
commuters          0
drive_alone        0
public_transit     0
walked             0
dtype: int64


,GEOID,population,median_income,housing_units,commuters,drive_alone,public_transit,walked
0,06037101110,4294.0,88438.0,1698.0,2232.0,1458.0,54.0,41.0
1,06037101122,4124.0,104643.0,1460.0,1992.0,1281.0,0.0,8.0
2,06037101220,3467.0,92317.0,1389.0,1806.0,1196.0,130.0,0.0
3,06037101221,3593.0,52204.0,1585.0,1495.0,1137.0,55.0,0.0
4,06037101222,2290.0,38125.0,1030.0,1067.0,694.0,74.0,93.0


## 4. Construct commuting shares and densities

I divide each commuting count by the B08301 total of workers age 16 and over, including those who work from home; the variable name `commuters` refers to that source total. A zero denominator remains missing rather than becoming an artificial zero share. These measures describe tract residents, not jobs located near the parking space.

In [5]:
acs["drive_alone_share"] = acs["drive_alone"] / acs["commuters"].where(acs["commuters"] > 0)
acs["transit_share"] = acs["public_transit"] / acs["commuters"].where(acs["commuters"] > 0)
acs["walk_share"] = acs["walked"] / acs["commuters"].where(acs["commuters"] > 0)
print(acs[["drive_alone_share", "transit_share", "walk_share"]].describe())

       drive_alone_share  transit_share   walk_share
count        2473.000000    2473.000000  2473.000000
mean            0.643520       0.042298     0.027185
std             0.120169       0.057867     0.060686
min             0.000000       0.000000     0.000000
25%             0.577413       0.006487     0.002331
50%             0.661463       0.022346     0.012627
75%             0.723684       0.055786     0.030488
max             1.000000       0.572973     1.000000


In [6]:
gazetteer = pd.read_csv(
    "../../data/raw/census/20261017T052111914317Z/2024_gaz_tracts_06.txt",
    sep="\t", dtype={"GEOID": str}
)
gazetteer.columns = gazetteer.columns.str.strip()
acs = acs.merge(gazetteer[["GEOID", "ALAND_SQMI"]], on="GEOID", how="left", indicator=True, validate="one_to_one")
print("Gazetteer matches:")
print(acs["_merge"].value_counts())
acs = acs.drop(columns="_merge")
print("Tracts with zero/nonpositive land area:", (acs["ALAND_SQMI"] <= 0).sum())
print("Tracts with missing land area:", acs["ALAND_SQMI"].isna().sum())

Gazetteer matches:
_merge
both          2498
left_only        0
right_only       0
Name: count, dtype: int64
Tracts with zero/nonpositive land area: 3
Tracts with missing land area: 0


Population density is people per square mile of tract land area, and housing density is housing units per square mile. I use Gazetteer `ALAND_SQMI` and leave densities missing when land area is zero or unavailable; the expected 2,495 nonmissing tract densities are a validation check, not grounds for imputing the others.

In [7]:
acs["population_density"] = acs["population"] / acs["ALAND_SQMI"].where(acs["ALAND_SQMI"] > 0)
acs["housing_density"] = acs["housing_units"] / acs["ALAND_SQMI"].where(acs["ALAND_SQMI"] > 0)
print("Nonmissing tract densities:")
print(acs[["population_density", "housing_density"]].count())
acs[["ALAND_SQMI", "population_density", "housing_density"]].describe()

Nonmissing tract densities:
population_density    2495
housing_density       2495
dtype: int64


,ALAND_SQMI,population_density,housing_density
count,2498.000000,2495.000000,2495.000000
mean,1.625408,13325.679910,5323.289743
std,15.216200,11310.653706,5967.552659
min,0.000000,0.000000,0.000000
25%,0.211000,6445.017127,2230.397727
50%,0.366500,10611.034483,3666.666667
75%,0.651750,17298.890633,6255.318236
max,575.175000,132736.842105,101157.894737


## 5. Load the official tract polygons

I use the official 2024 Census Bureau TIGER/Line California tract shapefile already downloaded for this project, then retain California state FIPS 06 and Los Angeles County FIPS 037. Gazetteer reference points are not tract boundaries, so they are not used to assign parking locations.

In [8]:
tracts_ca = gpd.read_file("../../data/raw/census/tigerline_2024_california_tracts/tl_2024_06_tract.shp")
tracts_la = tracts_ca.loc[(tracts_ca["STATEFP"] == "06") & (tracts_ca["COUNTYFP"] == "037")].copy()
print("California tracts:", len(tracts_ca))
print("Los Angeles County tracts:", len(tracts_la))
print("Source CRS:", tracts_la.crs)
print("Duplicate GEOIDs:", tracts_la["GEOID"].duplicated().sum())
print("Invalid tract geometries:", (~tracts_la.geometry.is_valid).sum())
tracts_la[["STATEFP", "COUNTYFP", "TRACTCE", "GEOID"]].head()

California tracts: 9129
Los Angeles County tracts: 2498
Source CRS: EPSG:4269
Duplicate GEOIDs: 0


Invalid tract geometries: 0


,STATEFP,COUNTYFP,TRACTCE,GEOID
2,06,037,204920,06037204920
3,06,037,205110,06037205110
4,06,037,205120,06037205120
5,06,037,206010,06037206010
6,06,037,206020,06037206020


## 6. Assign parking spaces to Census tracts

I first check the coordinates in the matched parking sample; the full inventory has known coordinate concerns, so they should not be assumed valid everywhere. Meter coordinates are treated as longitude/latitude in WGS84 (EPSG:4326), and I reproject the NAD83 tract polygons to that CRS before a `within` point-in-polygon join. This assigns actual containing tracts, with no nearest-centroid fallback.

In [9]:
plausible_coordinates = parking["latitude"].between(32, 35.5) & parking["longitude"].between(-120, -116)
print("Matched spaces with missing coordinates:", parking[["latitude", "longitude"]].isna().any(axis=1).sum())
print("Matched spaces outside broad Southern California bounds:", (~plausible_coordinates).sum())
print(parking.loc[~plausible_coordinates, ["SpaceID", "latitude", "longitude"]].head())
assert plausible_coordinates.all(), "Review implausible parking coordinates before geographic matching."

parking_points = gpd.GeoDataFrame(
    parking.copy(), geometry=gpd.points_from_xy(parking["longitude"], parking["latitude"]), crs="EPSG:4326"
)
tract_match = gpd.sjoin(parking_points, tracts_la[["GEOID", "geometry"]].to_crs(parking_points.crs), how="left", predicate="within")
print("Starting parking spaces:", len(parking_points))
print("Assigned tract:", tract_match["GEOID"].notna().sum())
print("Unmatched:", tract_match["GEOID"].isna().sum())
print("Match percentage:", tract_match["GEOID"].notna().mean() * 100)
print("Unique SpaceIDs:", tract_match["SpaceID"].nunique())
print("Duplicated SpaceIDs after join:", tract_match["SpaceID"].duplicated().sum())
print(tract_match.loc[tract_match["GEOID"].isna(), ["SpaceID", "latitude", "longitude"]].head(20))
assert not tract_match["SpaceID"].duplicated().any(), "Inspect multiple tract matches before continuing."
parking_geo = tract_match.drop(columns="index_right").copy()
parking_geo["tract_match_status"] = np.where(parking_geo["GEOID"].notna(), "matched", "unmatched")

Matched spaces with missing coordinates: 0
Matched spaces outside broad Southern California bounds: 0
Empty DataFrame
Columns: [SpaceID, latitude, longitude]
Index: []


Starting parking spaces: 4139
Assigned tract: 4139
Unmatched: 0
Match percentage: 100.0
Unique SpaceIDs: 4139
Duplicated SpaceIDs after join: 0
Empty DataFrame
Columns: [SpaceID, latitude, longitude]
Index: []


## 7. Attach the ACS candidate predictors

I merge the tract estimates onto parking spaces using the Census GEOID and check both linkage and missing estimates. All base spaces remain in the table even if a tract or estimate is missing; no Census values are imputed and no predictors are selected here.

In [10]:
parking_geo = parking_geo.merge(acs, on="GEOID", how="left", indicator=True, validate="many_to_one")
print("ACS merge status:")
print(parking_geo["_merge"].value_counts())
parking_geo["acs_match_status"] = np.where(parking_geo["_merge"] == "both", "matched", "unmatched")
parking_geo = parking_geo.drop(columns="_merge")
census_variables = ["population", "median_income", "housing_units", "population_density", "housing_density", "drive_alone_share", "transit_share", "walk_share"]
print("Missing Census predictors:")
print(parking_geo[census_variables].isna().sum())
print(parking_geo[census_variables].describe())
parking_geo[["SpaceID", "occupancy_rate", "price_midpoint", "GEOID"] + census_variables].head()

ACS merge status:
_merge
both          4139
left_only        0
right_only       0
Name: count, dtype: int64
Missing Census predictors:
population             0
median_income         44
housing_units          0
population_density     0
housing_density        0
drive_alone_share      0
transit_share          0
walk_share             0
dtype: int64


         population  median_income  housing_units  population_density  \
count   4139.000000    4095.000000    4139.000000         4139.000000   
mean    3449.185069   62970.026374    2357.271563        17841.532089   
std     2073.255205   29400.981266    1580.475393        12751.817083   
min     1051.000000    9375.000000       0.000000         1915.322581   
25%     1900.000000   46952.000000    1402.000000         9233.644860   
50%     2964.000000   52412.000000    1844.000000        14754.901961   
75%     4105.000000   69000.000000    2715.000000        26229.629630   
max    13681.000000  157115.000000    7590.000000       132736.842105   

       housing_density  drive_alone_share  transit_share   walk_share  
count      4139.000000        4139.000000    4139.000000  4139.000000  
mean      12190.224917           0.415807       0.094397     0.141054  
std        9814.547478           0.162707       0.061717     0.111217  
min           0.000000           0.088889       0.0000

,SpaceID,occupancy_rate,price_midpoint,GEOID,population,median_income,housing_units,population_density,housing_density,drive_alone_share,transit_share,walk_share
0,61111,0.515520,1.0,06037273404,1571.0,85688.0,1435.0,18702.380952,17083.333333,0.406043,0.0,0.022663
1,61112,0.468853,1.0,06037273404,1571.0,85688.0,1435.0,18702.380952,17083.333333,0.406043,0.0,0.022663
2,61113,0.540309,1.0,06037273404,1571.0,85688.0,1435.0,18702.380952,17083.333333,0.406043,0.0,0.022663
3,61114,0.594108,1.0,06037273404,1571.0,85688.0,1435.0,18702.380952,17083.333333,0.406043,0.0,0.022663
4,61116,0.524689,1.0,06037273404,1571.0,85688.0,1435.0,18702.380952,17083.333333,0.406043,0.0,0.022663


## 8. Inspect the business registry

The Office of Finance registry is an October 2026 snapshot, roughly fourteen months after the occupancy month. It is a proxy for commercial activity, not a verified August 2025 business census, and includes registered locations outside Los Angeles and locations without coordinates. I inspect the available fields, classifications, missing locations, and duplicates before constructing a count.

In [11]:
businesses = pd.read_csv(
    "../../data/raw/points_of_interest/20261017T052111914317Z/active_businesses.csv",
    dtype={"LOCATION ACCOUNT #": str, "NAICS": str}, low_memory=False
)
print("Raw business records:", len(businesses))
print("Columns:", businesses.columns.tolist())
print("Missing LOCATION:", businesses["LOCATION"].isna().sum())
print("Exact duplicate rows:", businesses.duplicated().sum())
print("Duplicate location account IDs:", businesses["LOCATION ACCOUNT #"].duplicated().sum())
print("Missing NAICS:", businesses["NAICS"].isna().sum())
print(businesses["PRIMARY NAICS DESCRIPTION"].value_counts().head(10))
businesses[["LOCATION ACCOUNT #", "NAICS", "PRIMARY NAICS DESCRIPTION", "LOCATION"]].head()

Raw business records: 632799
Columns: ['LOCATION ACCOUNT #', 'BUSINESS NAME', 'DBA NAME', 'STREET ADDRESS', 'CITY', 'ZIP CODE', 'LOCATION DESCRIPTION', 'MAILING ADDRESS', 'MAILING CITY', 'MAILING ZIP CODE', 'NAICS', 'PRIMARY NAICS DESCRIPTION', 'COUNCIL DISTRICT', 'LOCATION START DATE', 'LOCATION END DATE', 'LOCATION']
Missing LOCATION: 66505


Exact duplicate rows: 0
Duplicate location account IDs: 0
Missing NAICS: 57281
PRIMARY NAICS DESCRIPTION
Lessors of real estate (including mini warehouses & self-storage units)                                                                                           42258
All other personal services                                                                                                                                       40857
Independent artists, writers, & performers                                                                                                                        34511
Single Family Housing Construction (1997 NAICS)                                                                                                                   16322
All other professional, scientific, & technical services                                                                                                          15785
Management, scientific, & technical consulting services

,LOCATION ACCOUNT #,NAICS,PRIMARY NAICS DESCRIPTION,LOCATION
0,0000000320-0019-3,NaN,NaN,"(34.0386, -118.2197)"
1,0000000321-0001-5,NaN,NaN,NaN
2,0000000554-0001-6,NaN,NaN,"(34.1017, -118.305)"
3,0000001036-0012-0,621400,Outpatient care centers,"(34.0366, -118.2461)"
4,0000001333-0001-5,611000,"Educational services (including schools, colle...",NaN


## 9. Prepare located businesses and a transparent activity subset

I retain distinct source records with usable latitude/longitude, without collapsing different business accounts at the same address. Missing or invalid locations cannot contribute to a radius count, but this does not imply those businesses do not exist. The combined restaurant/retail candidate uses NAICS prefixes 722 (food services and drinking places, broader than restaurants alone) and 44–45 (retail); missing classifications contribute only to the all-business count.

In [12]:
businesses = businesses.drop_duplicates().copy()
business_coordinates = businesses["LOCATION"].str.extract(r"\(\s*([-0-9.]+)\s*,\s*([-0-9.]+)\s*\)")
businesses["business_latitude"] = pd.to_numeric(business_coordinates[0], errors="coerce")
businesses["business_longitude"] = pd.to_numeric(business_coordinates[1], errors="coerce")
usable_locations = (
    businesses["business_latitude"].between(-90, 90)
    & businesses["business_longitude"].between(-180, 180)
    & ~((businesses["business_latitude"] == 0) & (businesses["business_longitude"] == 0))
)
print("Distinct business records:", len(businesses))
print("Records with usable locations:", usable_locations.sum())
print("Excluded from spatial counting for location issues:", (~usable_locations).sum())
businesses_located = businesses.loc[usable_locations].copy()
businesses_located["restaurant_retail"] = businesses_located["NAICS"].fillna("").str.startswith(("722", "44", "45"))
print("Located food-service/retail records:", businesses_located["restaurant_retail"].sum())
business_points = gpd.GeoDataFrame(
    businesses_located[["LOCATION ACCOUNT #", "restaurant_retail"]].copy(),
    geometry=gpd.points_from_xy(businesses_located["business_longitude"], businesses_located["business_latitude"]), crs="EPSG:4326"
)

Distinct business records: 632799
Records with usable locations: 559615
Excluded from spatial counting for location issues: 73184
Located food-service/retail records: 65528


## 10. Count businesses within a quarter mile

I choose a fixed 0.25-mile radius (402.336 metres) as a local walking-distance measure, without using model performance to choose it. Parking and business points are projected to NAD83 / UTM zone 11N (EPSG:26911), whose metre units are suitable for distances around Los Angeles. A zero means no eligible geocoded registry records were found in the buffer, not proof that there are no actual businesses; the registry's missing locations and later vintage limit interpretation.

In [13]:
radius_miles = 0.25
radius_metres = radius_miles * 1609.344
parking_projected = parking_geo.to_crs("EPSG:26911")
business_projected = business_points.to_crs(parking_projected.crs)
finite_business_points = np.isfinite(business_projected.geometry.x) & np.isfinite(business_projected.geometry.y)
print("Nonfinite projected business locations:", (~finite_business_points).sum())
business_projected = business_projected.loc[finite_business_points].copy()
parking_buffers = parking_projected[["SpaceID", "geometry"]].copy()
parking_buffers["geometry"] = parking_buffers.geometry.buffer(radius_metres)
business_matches = gpd.sjoin(parking_buffers, business_projected, how="left", predicate="intersects")
business_counts = business_matches.groupby("SpaceID").agg(
    nearby_business_count=("index_right", "count"),
    nearby_restaurant_retail_count=("restaurant_retail", "sum")
)
parking_geo = parking_geo.merge(business_counts, on="SpaceID", how="left", validate="one_to_one")
parking_geo[["nearby_business_count", "nearby_restaurant_retail_count"]] = parking_geo[["nearby_business_count", "nearby_restaurant_retail_count"]].astype("int64")
print("Radius in miles / metres:", radius_miles, radius_metres)
print("Parking spaces receiving counts:", parking_geo["nearby_business_count"].notna().sum())
print(parking_geo[["nearby_business_count", "nearby_restaurant_retail_count"]].describe())
parking_geo[["SpaceID", "nearby_business_count", "nearby_restaurant_retail_count"]].head(10)

Nonfinite projected business locations: 0


Radius in miles / metres: 0.25 402.336
Parking spaces receiving counts: 4139
       nearby_business_count  nearby_restaurant_retail_count
count            4139.000000                     4139.000000
mean             1177.134332                      228.786422
std               704.700473                      148.293265
min               129.000000                       21.000000
25%               678.500000                      119.000000
50%               949.000000                      176.000000
75%              1592.500000                      334.000000
max              4825.000000                      878.000000


,SpaceID,nearby_business_count,nearby_restaurant_retail_count
0,61111,358,45
1,61112,368,46
2,61113,371,47
3,61114,371,46
4,61116,375,47
5,61121,402,52
6,61122,420,53
7,61123,437,55
8,61124,444,55
9,61125,450,55


## 11. Read and validate the zoning batches

The saved ArcGIS JSON pages contain zoning attributes and polygon rings in EPSG:3857. I convert the ring linework to polygon areas so holes and separate parts are represented, then inspect source IDs, duplicate features, and geometry validity.

Two unreadable original pages require separately saved current supplements; two other pages were restored with their original hashes. I keep the original distinct-ID coverage but disclose changed duplicate multiplicities and mixed retrieval dates in the checks below; full recovery provenance is in the [completion report](../../docs/notebook03_completion_report.md). Raw supplemental responses are not edited.

In [14]:
original_zoning_files = sorted(Path("../../data/raw/land_use").glob("20261017*/zoning_offset_*.json"))
supplement_folder = Path("../../data/raw/land_use/supplement_20261022")
supplement_names = ["zoning_offset_035000.json", "zoning_offset_055000.json"]
zoning_files = [supplement_folder / path.name if path.name in supplement_names else path for path in original_zoning_files]
zoning_attributes = []
zoning_geometry = []
zoning_reference = []
for path in zoning_files:
    with path.open() as f:
        batch = json.load(f)
    zoning_reference.append(batch["spatialReference"])
    for feature in batch["features"]:
        zoning_attributes.append(feature["attributes"])
        zoning_geometry.append(build_area(MultiLineString(feature["geometry"]["rings"])))
zoning = gpd.GeoDataFrame(zoning_attributes, geometry=zoning_geometry, crs="EPSG:3857")
print("Zoning pages:", len(zoning_files))
print("Raw feature count:", len(zoning))
with Path("../../data/raw/land_use/20261017T052654015392Z/zoning_source_ids.json").open() as f:
    original_zoning_ids = json.load(f)["objectIds"]
print("New source IDs outside the original listing:", (~zoning["OBJECTID"].isin(original_zoning_ids)).sum())
zoning = zoning.loc[zoning["OBJECTID"].isin(original_zoning_ids)].copy()
print("Features within the original ID coverage:", len(zoning))
original_id_counts = Counter(original_zoning_ids)
working_id_counts = Counter(zoning["OBJECTID"])
print("Source IDs with changed multiplicities:", sum(working_id_counts[k] != original_id_counts[k] for k in original_id_counts))
print("All original distinct IDs represented:", set(working_id_counts) == set(original_id_counts))
assert set(working_id_counts) == set(original_id_counts)
print("Source spatial references:", {str(v) for v in zoning_reference})
print("Fields:", zoning.columns.tolist())
print("Geometry types:")
print(zoning.geom_type.value_counts())
print("Repeated source IDs:", zoning["OBJECTID"].duplicated().sum())
print("Exact duplicate features:", zoning.duplicated().sum())
print("Empty geometries:", zoning.geometry.is_empty.sum())
print("Invalid geometries:", (~zoning.geometry.is_valid).sum())
zoning.loc[zoning["OBJECTID"].duplicated(keep=False), ["OBJECTID", "ZONE_CMPLT", "ZONE_CLASS", "ZONING_DESCRIPTION"]].head(12)

Zoning pages: 28
Raw feature count: 58956
New source IDs outside the original listing: 4
Features within the original ID coverage: 58952
Source IDs with changed multiplicities: 40
All original distinct IDs represented: True
Source spatial references: {"{'wkid': 102100, 'latestWkid': 3857}"}
Fields: ['OBJECTID', 'ZONE_CMPLT', 'ZONE_UNDER', 'JTX_ID', 'PAPER_ST', 'ZONE_CLASS', 'ZONE_CODE', 'ZONING_DESCRIPTION', 'TOOLTIP', 'geometry']
Geometry types:
Polygon               58903
MultiPolygon             34
GeometryCollection       15
Name: count, dtype: int64
Repeated source IDs: 97


Exact duplicate features: 0
Empty geometries: 15
Invalid geometries: 0


,OBJECTID,ZONE_CMPLT,ZONE_CLASS,ZONING_DESCRIPTION
588,589,UV(CA),UV(CA),"Manufacturing/Industrial - CM,MR,CCS,M1,WC,M2,..."
589,589,UV(CA),,Industrial-Mixed (Chapter 1A)
2609,2609,UV(CA),UV(CA),"Manufacturing/Industrial - CM,MR,CCS,M1,WC,M2,..."
2610,2609,UV(CA),,Industrial-Mixed (Chapter 1A)
2613,2612,UV(CA),UV(CA),"Manufacturing/Industrial - CM,MR,CCS,M1,WC,M2,..."
2614,2612,UV(CA),,Industrial-Mixed (Chapter 1A)
3548,3546,UV(CA),UV(CA),"Manufacturing/Industrial - CM,MR,CCS,M1,WC,M2,..."
3549,3546,UV(CA),,Industrial-Mixed (Chapter 1A)
3550,3547,UV(CA),UV(CA),"Manufacturing/Industrial - CM,MR,CCS,M1,WC,M2,..."
3551,3547,UV(CA),,Industrial-Mixed (Chapter 1A)


## 12. Match the available zoning polygons

I check for exact duplicate features while retaining distinct records that share a source ID. Parking spaces lie along streets while zoning polygons may describe adjacent parcels, so unmatched points do not receive a neighbor's zoning by assumption. I preserve source labels and leave out the 15 empty polygon areas after displaying their attributes; no boundary is invented and no parking observation is removed.

In [15]:
zoning = zoning.drop_duplicates().copy()
print("Features after exact duplicate removal:", len(zoning))
print("Remaining repeated source IDs:", zoning["OBJECTID"].duplicated().sum())
print("Source descriptions:")
print(zoning["ZONING_DESCRIPTION"].value_counts().head(15))
assert zoning.geometry.is_valid.all(), "Inspect invalid zoning geometry before spatial matching."
print("Source records with empty polygon areas:")
print(zoning.loc[zoning.geometry.is_empty, ["OBJECTID", "ZONE_CMPLT", "ZONE_CLASS"]].head(15))
zoning = zoning.loc[~zoning.geometry.is_empty].copy()
print("Nonempty features used for matching:", len(zoning))
zoning_match = gpd.sjoin(
    parking_geo[["SpaceID", "geometry"]].to_crs(zoning.crs),
    zoning[["ZONE_CMPLT", "ZONE_CLASS", "ZONING_DESCRIPTION", "geometry"]],
    how="left", predicate="within"
)
print("Rows after zoning join:", len(zoning_match))
print("Extra repeated SpaceIDs:", zoning_match["SpaceID"].duplicated().sum())
print("Spaces with multiple polygon matches:", zoning_match.loc[zoning_match["SpaceID"].duplicated(keep=False), "SpaceID"].nunique())
zoning_match.loc[zoning_match["SpaceID"].duplicated(keep=False), ["SpaceID", "ZONE_CMPLT", "ZONE_CLASS", "ZONING_DESCRIPTION"]].head(20)

Features after exact duplicate removal: 58952
Remaining repeated source IDs: 97
Source descriptions:
ZONING_DESCRIPTION
Residential - RE,RS,R1,RU,RZ,RW1                             20053
Multiple Dwelling - R2,RD,RMP,RW2,R3,R4,R5                   15822
Commercial - ADP, C1, C1.5, C2, C4, C5, CR, CW, LASED, WC    10571
Manufacturing/Industrial - CM,MR,CCS,M1,WC,M2,M3,SL           3107
Agricultural/Suburvan - A,RA                                  2441
Public Facility - PF                                          1913
Open Space - OS                                               1442
Parking - P,PB                                                1304
Commercial-Mixed (Chapter 1A)                                  752
Residential-Mixed (Chapter 1A)                                 520
Industrial-Mixed (Chapter 1A)                                  427
Industrial (Chapter 1A)                                        192
Public (Chapter 1A)                                            183
Residenti

,SpaceID,ZONE_CMPLT,ZONE_CLASS,ZONING_DESCRIPTION


If overlapping polygons agree on code, class, and description, they imply the same assignment; conflicting labels remain missing and are flagged rather than resolved by taking the first polygon. Blank source labels are recorded as missing. Only 49 spaces (1.18%) receive a source code, so zoning is not a broadly usable predictor for the main sample, and no nearest-polygon fallback or new broad category is applied.

In [16]:
zoning_labels = zoning_match[["SpaceID", "ZONE_CMPLT", "ZONE_CLASS", "ZONING_DESCRIPTION"]].drop_duplicates()
ambiguous_zoning_ids = zoning_labels.loc[zoning_labels["SpaceID"].duplicated(keep=False), "SpaceID"].unique()
print("Spaces with conflicting zoning labels:", len(ambiguous_zoning_ids))
print(zoning_labels.loc[zoning_labels["SpaceID"].isin(ambiguous_zoning_ids)].head(20))
zoning_labels = zoning_labels.loc[~zoning_labels["SpaceID"].isin(ambiguous_zoning_ids)].copy()
zoning_labels = zoning_labels.rename(columns={"ZONE_CMPLT": "zoning_code", "ZONE_CLASS": "zoning_class", "ZONING_DESCRIPTION": "zoning_description"})
zoning_labels[["zoning_code", "zoning_class", "zoning_description"]] = zoning_labels[["zoning_code", "zoning_class", "zoning_description"]].replace(r"^\s*$", pd.NA, regex=True)
parking_geo = parking_geo.merge(zoning_labels, on="SpaceID", how="left", validate="one_to_one")
parking_geo["zoning_match_status"] = np.where(parking_geo["zoning_code"].notna(), "matched", "unmatched")
parking_geo.loc[parking_geo["SpaceID"].isin(ambiguous_zoning_ids), "zoning_match_status"] = "ambiguous"
print("Zoning match status:")
print(parking_geo["zoning_match_status"].value_counts())
print("Assigned zoning percentage:", (parking_geo["zoning_match_status"] == "matched").mean() * 100)
print("Duplicate parking IDs:", parking_geo["SpaceID"].duplicated().sum())
print(parking_geo["zoning_code"].value_counts().head(10))
print(parking_geo["zoning_description"].value_counts().head(10))
parking_geo.loc[parking_geo["zoning_match_status"] != "matched", ["SpaceID", "latitude", "longitude", "zoning_match_status"]].head(15)

Spaces with conflicting zoning labels: 0
Empty DataFrame
Columns: [SpaceID, ZONE_CMPLT, ZONE_CLASS, ZONING_DESCRIPTION]
Index: []
Zoning match status:
zoning_match_status
unmatched    4090
matched        49
Name: count, dtype: int64
Assigned zoning percentage: 1.1838608359507128
Duplicate parking IDs: 0
zoning_code
[DM1-G1-5][P2-FA][CPIO-O]     16
C2-2D-CPIO                    11
[HB3-G1-5][CX2-FA][CPIO-O]     8
C4-1                           6
[DM1-AL1-5][IX3-FA][CPIO]      3
[HB5-SH1-5][CX3-FA][CPIO]      1
[DM1-SH1-5][IX3-FA][CPIO]      1
VARIOUS                        1
PF-2D                          1
[Q]C2-2D-CPIO                  1
Name: count, dtype: int64
zoning_description
Commercial - ADP, C1, C1.5, C2, C4, C5, CR, CW, LASED, WC    18
Public (Chapter 1A)                                          16
Commercial-Mixed (Chapter 1A)                                 9
Industrial-Mixed (Chapter 1A)                                 4
Public Facility - PF                                

,SpaceID,latitude,longitude,zoning_match_status
0,61111,33.997217,-118.478897,unmatched
1,61112,33.997177,-118.478857,unmatched
2,61113,33.997106,-118.478788,unmatched
3,61114,33.997058,-118.478741,unmatched
4,61116,33.996964,-118.478645,unmatched
5,61121,33.996677,-118.478366,unmatched
6,61122,33.996625,-118.478314,unmatched
7,61123,33.996580,-118.478269,unmatched
8,61124,33.996527,-118.478216,unmatched
9,61125,33.996485,-118.478172,unmatched


## 13. Assemble one row per parking space

I keep the original outcome and support variables, meter fields, Census candidates, two business counts, and source zoning labels. Tract land area and commuting counts remain as traceable components of the derived predictors, and match-status fields explain missing spatial data. No complete-case restriction or final predictor selection is applied.

In [17]:
final_columns = [
    "SpaceID", "occupancy_rate", "occupied_minutes", "total_minutes", "active_days",
    "BlockFace", "MeterType", "RateType", "RateRange", "MeteredTimeLimit",
    "price_midpoint", "time_limit_hours", "latitude", "longitude", "GEOID",
    "population", "median_income", "housing_units", "commuters", "drive_alone", "public_transit", "walked",
    "ALAND_SQMI", "population_density", "housing_density", "drive_alone_share", "transit_share", "walk_share",
    "nearby_business_count", "nearby_restaurant_retail_count", "zoning_code", "zoning_class", "zoning_description",
    "tract_match_status", "acs_match_status", "zoning_match_status"
]
parking_cross_section = pd.DataFrame(parking_geo[final_columns]).copy()
print("Final rows and columns:", parking_cross_section.shape)
print("Unique SpaceIDs:", parking_cross_section["SpaceID"].nunique())
print("Duplicate SpaceIDs:", parking_cross_section["SpaceID"].duplicated().sum())
assert len(parking_cross_section) == len(parking)
assert not parking_cross_section["SpaceID"].duplicated().any()
parking_cross_section.head()

Final rows and columns: (4139, 36)
Unique SpaceIDs: 4139
Duplicate SpaceIDs: 0


,SpaceID,occupancy_rate,occupied_minutes,total_minutes,active_days,BlockFace,MeterType,RateType,RateRange,MeteredTimeLimit,...,transit_share,walk_share,nearby_business_count,nearby_restaurant_retail_count,zoning_code,zoning_class,zoning_description,tract_match_status,acs_match_status,zoning_match_status
0,61111,0.515520,19511.316667,37847.833333,30,245 MAIN ST,Multi-Space,TOD,$0.50 - $1.50,2HR,...,0.0,0.022663,358,45,NaN,NaN,NaN,matched,matched,unmatched
1,61112,0.468853,17954.033333,38293.550000,30,245 MAIN ST,Multi-Space,TOD,$0.50 - $1.50,2HR,...,0.0,0.022663,368,46,NaN,NaN,NaN,matched,matched,unmatched
2,61113,0.540309,21939.050000,40604.650000,30,245 MAIN ST,Multi-Space,TOD,$0.50 - $1.50,2HR,...,0.0,0.022663,371,47,NaN,NaN,NaN,matched,matched,unmatched
3,61114,0.594108,26049.100000,43845.700000,31,245 MAIN ST,Multi-Space,TOD,$0.50 - $1.50,2HR,...,0.0,0.022663,371,46,NaN,NaN,NaN,matched,matched,unmatched
4,61116,0.524689,22993.916667,43823.916667,31,245 MAIN ST,Multi-Space,TOD,$0.50 - $1.50,2HR,...,0.0,0.022663,375,47,NaN,NaN,NaN,matched,matched,unmatched


## 14. Validate the candidate variables

I check missingness, ranges, and category counts to confirm what was constructed and what remains unresolved. Large values alone do not trigger exclusions here, and missing predictors are preserved for Notebook 04. These checks do not choose a regression specification or estimate any model.

In [18]:
candidate_variables = [
    "price_midpoint", "time_limit_hours", "MeterType", "RateType", "latitude", "longitude",
    "population", "median_income", "housing_units", "population_density", "housing_density",
    "drive_alone_share", "transit_share", "walk_share", "nearby_business_count",
    "nearby_restaurant_retail_count", "zoning_code", "zoning_class", "zoning_description"
]
missingness = pd.DataFrame({
    "missing_count": parking_cross_section[candidate_variables].isna().sum(),
    "missing_percent": parking_cross_section[candidate_variables].isna().mean() * 100
})
print(missingness)
numeric_candidates = parking_cross_section[candidate_variables].select_dtypes(include="number").columns.tolist()
print(parking_cross_section[["occupancy_rate"] + numeric_candidates].describe().T)
for variable in ["MeterType", "RateType", "zoning_class", "zoning_description"]:
    print("\n", variable)
    print(parking_cross_section[variable].value_counts(dropna=False).head(12))

                                missing_count  missing_percent
price_midpoint                              0         0.000000
time_limit_hours                            0         0.000000
MeterType                                   0         0.000000
RateType                                    0         0.000000
latitude                                    0         0.000000
longitude                                   0         0.000000
population                                  0         0.000000
median_income                              44         1.063059
housing_units                               0         0.000000
population_density                          0         0.000000
housing_density                             0         0.000000
drive_alone_share                           0         0.000000
transit_share                               0         0.000000
walk_share                                  0         0.000000
nearby_business_count                       0         0

In [19]:
print("Occupancy outside [0, 1]:", (~parking_cross_section["occupancy_rate"].between(0, 1)).sum())
print("Latitude range:", parking_cross_section["latitude"].min(), parking_cross_section["latitude"].max())
print("Longitude range:", parking_cross_section["longitude"].min(), parking_cross_section["longitude"].max())
for variable in ["drive_alone_share", "transit_share", "walk_share"]:
    values = parking_cross_section[variable].dropna()
    print(variable, "range:", values.min(), values.max(), "outside [0, 1]:", (~values.between(0, 1)).sum())
nonnegative_variables = [
    "occupied_minutes", "total_minutes", "active_days", "price_midpoint", "time_limit_hours",
    "population", "median_income", "housing_units", "commuters", "drive_alone", "public_transit", "walked",
    "ALAND_SQMI", "population_density", "housing_density", "nearby_business_count", "nearby_restaurant_retail_count"
]
print("Negative values in nonnegative quantities:")
print((parking_cross_section[nonnegative_variables] < 0).sum())
print("Nonfinite nonmissing numeric values:", np.isinf(parking_cross_section.select_dtypes(include="number")).sum().sum())
print("Business subset count above all-business count:", (parking_cross_section["nearby_restaurant_retail_count"] > parking_cross_section["nearby_business_count"]).sum())
assert parking_cross_section["occupancy_rate"].between(0, 1).all()
assert parking_cross_section["active_days"].ge(25).all()

Occupancy outside [0, 1]: 0
Latitude range: 33.736298 34.105161
Longitude range: -118.480323 -118.23611
drive_alone_share range: 0.08888888888888889 0.7903225806451613 outside [0, 1]: 0
transit_share range: 0.0 0.27111111111111114 outside [0, 1]: 0
walk_share range: 0.0 0.6271929824561403 outside [0, 1]: 0
Negative values in nonnegative quantities:
occupied_minutes                  0
total_minutes                     0
active_days                       0
price_midpoint                    0
time_limit_hours                  0
population                        0
median_income                     0
housing_units                     0
commuters                         0
drive_alone                       0
public_transit                    0
walked                            0
ALAND_SQMI                        0
population_density                0
housing_density                   0
nearby_business_count             0
nearby_restaurant_retail_count    0
dtype: int64
Nonfinite nonmissing num

## 15. Save the cross-section for later analysis

I save the one-row-per-space dataset with missing predictors and match-status information intact. The occupancy outcome and parking variables have not been redefined, and price remains a proxy with potential endogeneity. EDA, predictor selection, OLS, diagnostics, and model-based transformations belong in later notebooks.

In [20]:
output_path = Path("../../data/processed/parking_cross_section.csv")
parking_cross_section.to_csv(output_path, index=False)
print("Saved shape:", parking_cross_section.shape)
print("Saved path:", output_path)
parking_cross_section[["SpaceID", "occupancy_rate", "price_midpoint", "GEOID", "population_density", "nearby_business_count", "zoning_class"]].head()

Saved shape: (4139, 36)
Saved path: ../data/processed/parking_cross_section.csv


,SpaceID,occupancy_rate,price_midpoint,GEOID,population_density,nearby_business_count,zoning_class
0,61111,0.515520,1.0,06037273404,18702.380952,358,NaN
1,61112,0.468853,1.0,06037273404,18702.380952,368,NaN
2,61113,0.540309,1.0,06037273404,18702.380952,371,NaN
3,61114,0.594108,1.0,06037273404,18702.380952,371,NaN
4,61116,0.524689,1.0,06037273404,18702.380952,375,NaN


## Summary

The saved cross-section retains the 4,139 spaces that match the cleaned meter inventory; Census and business linkage is complete, but source zoning codes match only 49 spaces (1.18%) and remain an incomplete candidate rather than a ready general land-use control. Missing or conflicting spatial assignments remain visible rather than being silently dropped or imputed.

The quarter-mile business radius is researcher-selected, the business and meter snapshots are later than August 2025, and zoning polygons may not cover street parking points. The Census release summarizes 2020–2024, while the outcome is August 2025; all coefficients in later OLS work should be interpreted as associations unless a stronger identification argument is developed.